# ContractRisk — juez 13: confirmación independiente

Runtime CPU. Ejecuta SOLO los doce ejemplos sintéticos de confirmación aprobados por el
equipo antes de esta corrida. No repite los 38 juicios de calibración ni regenera respuestas
contractuales. No recibe gold contractual y no cambia rúbrica, modelo ni configuración.

Reglas fijadas: 12/12 juicios válidos, al menos 11/12 dentro del rango esperado y ninguna
respuesta mala aprobada con 4 o 5. Es una comprobación adicional en esta tarea, no garantía
universal de buen juicio. Incluso si pasa, los motivos deben auditarse antes de validar al juez.

Usa el mismo proyecto de prepago y la misma cuenta de Drive. Recarga automática desactivada,
ningún otro uso de esta API durante la corrida. Techo USD 5 TOTALES, no adicionales.
Registro inicial: 45 asientos por USD 4,0800825 estimados/reservados; margen local
USD 0,9199175, que no es saldo ni factura de Google. Se conserva toda reserva desconocida.

No hay reintentos automáticos ante 503, timeout o red. Un formato inválido admite una sola
reparación bajo el presupuesto; nunca se repite una nota válida para mejorarla. Se detiene
ante fallo de generación/formato/presupuesto o si aprueba una respuesta mala con 4–5.
Puede detenerse sin completar los doce si no queda margen para la siguiente reserva.

Ejecutar una sola vez, en orden, con GEMINI_API_KEY y WANDB_API_KEY habilitados como secretos.
No comprar más créditos, borrar registros, cambiar referencias ni reintentar si falla.
Si se detiene, ir únicamente a inspección y descarga. Guardar ZIP y notebook con salidas.
Tarifas verificadas 19/11/2026; después de 20/11/2026 UTC exige revisión de precios.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_confirmation13_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '71912f1c5755790381a8e39c8a67801bcbf9243d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '7afdda0212bf38ad1accd043a968db722569c618'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'ece58275c2a19abfb6bbd2fe4b507731232e9083'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '4c5d971e95d4a3d6195c79c542154a8db2dcd6e0'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
confirmation = input('Confirma mismo proyecto de prepago, recarga automática desactivada, ninguna otra aplicación/usuario usando esta API y techo TOTAL USD 5. Escribe CONFIRMACION_MAXIMO_5_USD: ')
assert confirmation.strip() == 'CONFIRMACION_MAXIMO_5_USD', 'Sin confirmación no se consulta la API.'
del confirmation
from final.hydrogen_confirmation13 import verify_checkpoint, verify_approval, check_drive
verify_approval(ROOT)
checkpoint = verify_checkpoint(ROOT)
cfg = json.loads((ROOT / 'final/hydrogen_calibration_config.json').read_text(encoding='utf-8'))
check_drive(BUDGET_DIR, cfg, checkpoint)
print('Doce casos aprobados, calibración previa, 45 asientos y seis marcadores verificados sin consultas de modelo.')


In [ ]:
RESULTS = ROOT / 'confirmation13_results'
run_visible([sys.executable, '-u', 'final/run_hydrogen_confirmation13.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-paid-project', '--confirm-exclusive-budget-use',
    '--require-wandb'], 'confirmation13.log')


## Inspección y conservación

Un resultado favorable de confirmación no autoriza automáticamente puntuar el producto.
Falta revisar los motivos, conciliar el presupuesto y preparar la evaluación posterior.
No mezclar doce juicios de confirmación con las 38 calificaciones históricas.
Si falla, conservar resultados y no repetir, borrar marcadores ni aumentar el presupuesto.


In [ ]:
RESULTS = ROOT / 'confirmation13_results'
for name in ['metadata.json', 'confirmation_summary.json']:
    path = RESULTS / name
    if path.exists(): print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
from final.hydrogen_confirmation13 import POLICY
RESULTS = ROOT / 'confirmation13_results'
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
(RESULTS / 'confirmation13_policy.json').write_text(json.dumps(POLICY, indent=2), encoding='utf-8')
for logfile in ROOT.glob('*.log'): shutil.copy2(logfile, RESULTS / logfile.name)
pack = RESULTS / 'approved_confirmation_pack'
if not pack.exists(): shutil.copytree(ROOT / 'final/eval/confirmation_v1', pack)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_confirmacion13_resultados'), 'zip', RESULTS)
files.download(package)
